# Chapitre 18 - Problèmes en grande dimension ($p \gg N$)
## Cours 3 : ACP supervisée, tests multiples et taux de fausses découvertes

Plan (sections 18.6 et 18.7 du livre) :

1. **composantes principales supervisées** : régression en grande dimension via une
   variable latente ; comparaison avec la PLS, et « pré-conditionnement » du lasso ;
2. **évaluation des variables** et problème des **tests multiples** : FWER, Bonferroni,
   taux de fausses découvertes (FDR), procédure de **Benjamini-Hochberg**, estimation
   « plug-in » par permutations, procédure **SAM**, interprétation bayésienne.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from esl.plotting import set_style, COLORS, BLUE, ORANGE, GRAY, SEQ, DIV, TWO_CLASS
set_style()

from esl import datasets
from esl import ch18_grande_dimension as g

## 1. Composantes principales supervisées (section 18.6)

### 1.1 Le modèle à variable latente

Le livre motive la méthode par un exemple de survie (lymphomes) : on imagine deux
« types cellulaires » ; les patients du bon type vivent plus longtemps en moyenne.
Le type cellulaire n'est pas observé, mais il se reflète dans l'expression de tout un
**groupe** de gènes (une voie biologique). Formellement (éq. (18.32)-(18.33)) :
$$Y = \beta_0 + \beta_1 U + \varepsilon, \qquad X_j = \alpha_{0j} + \alpha_{1j}U + \epsilon_j,\quad j\in\mathcal P,$$
où $U$ est une **variable latente** (le « type cellulaire » continu), $\mathcal P$
l'ensemble des gènes de la voie, et les autres gènes $X_k$, $k\notin\mathcal P$, sont
indépendants de $U$. Les erreurs sont centrées et indépendantes de tout le reste.

> **Intuition.** Chaque gène de $\mathcal P$ est une **mesure bruitée** de $U$. En
> moyennant beaucoup de ces mesures (c'est ce que fait la première composante
> principale de $\mathbf X_{\mathcal P}$), on estime $U$ bien mieux qu'avec un seul gène,
> puis on régresse $Y$ sur cette estimation.

### 1.2 L'algorithme (algorithme 18.1)

1. Pour chaque variable $j$, calculer le coefficient de régression **univarié
   standardisé** de $y$ sur $x_j$ (une statistique $t$).
2. Pour chaque seuil $\theta$ d'une grille $0\le\theta_1<\dots<\theta_K$ :
   - (a) ne garder que les variables dont $|$score$| > \theta$ et calculer les $m$
     premières composantes principales de cette matrice réduite ;
   - (b) utiliser ces composantes dans un modèle de régression pour prédire $y$.
3. Choisir $\theta$ (et $m$) par validation croisée.

Lien avec le modèle latent : l'étape 1 estime $\mathcal P$, l'étape 2a estime $U$, et
l'étape 2b estime $\beta_0,\beta_1$. Le plus souvent $m = 1$.